# Ad Spend Analysis: SQL
Client question: which channels drive revenue, where has spend hit diminishing returns, and how should next year's budget move?

In [2]:
import duckdb
con = duckdb.connect("../data/ads.duckdb")

for t in ["campaigns", "daily_performance", "weekly_sales", "promotions"]:
    con.sql(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv_auto('../data/{t}.csv')")

con.sql("SHOW TABLES")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ campaigns         │
│ daily_performance │
│ promotions        │
│ weekly_sales      │
└───────────────────┘

In [4]:
con.sql("DESCRIBE daily_performance").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,campaign_id,BIGINT,YES,None,None,None
2,spend,DOUBLE,YES,None,None,None
3,impressions,BIGINT,YES,None,None,None
4,clicks,BIGINT,YES,None,None,None
5,conversions,BIGINT,YES,None,None,None
6,revenue,DOUBLE,YES,None,None,None


In [5]:
con.sql("DESCRIBE weekly_sales").df()

,column_name,column_type,null,key,default,extra
0,week_start,DATE,YES,None,None,None
1,total_sales,DOUBLE,YES,None,None,None


In [6]:
con.sql("DESCRIBE promotions").df()

,column_name,column_type,null,key,default,extra
0,week_start,DATE,YES,None,None,None
1,promo_name,VARCHAR,YES,None,None,None
2,discount_pct,BIGINT,YES,None,None,None


In [7]:
con.sql("DESCRIBE campaigns").df()

,column_name,column_type,null,key,default,extra
0,campaign_id,BIGINT,YES,None,None,None
1,campaign_name,VARCHAR,YES,None,None,None
2,channel,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,language,VARCHAR,YES,None,None,None
5,creative_type,VARCHAR,YES,None,None,None
6,objective,VARCHAR,YES,None,None,None
7,start_date,DATE,YES,None,None,None
8,end_date,DATE,YES,None,None,None


# A. Data check

In [10]:
con.sql("""
SELECT 'campaigns' AS tbl, COUNT(*) AS n_rows, MIN(start_date) AS first_date, MAX(end_date) AS last_date FROM campaigns
UNION ALL
SELECT 'daily_performance', COUNT(*), MIN(date), MAX(date) FROM daily_performance
UNION ALL
SELECT 'weekly_sales', COUNT(*), MIN(week_start), MAX(week_start) FROM weekly_sales
UNION ALL
SELECT 'promotions', COUNT(*), MIN(week_start), MAX(week_start) FROM promotions
""").df()

,tbl,n_rows,first_date,last_date
0,campaigns,229,2024-01-01,2025-12-28
1,daily_performance,13062,2024-01-01,2025-12-28
2,weekly_sales,104,2024-01-01,2025-12-22
3,promotions,12,2024-02-12,2025-11-24


## Findings
- All four tables cover 2024-01-01 to 2025-12-28. weekly_sales has 104 rows = 104 weeks, no gaps.
- promotions has only 12 rows: it lists promo weeks only. Join it to weekly_sales with a LEFT JOIN, otherwise non-promo weeks are dropped.

In [11]:
con.sql("""
SELECT date, campaign_id, COUNT(*) AS n
FROM daily_performance
GROUP BY date, campaign_id
HAVING COUNT(*) > 1
""").df()

,date,campaign_id,n
0,2025-03-04,1137,2
1,2025-03-15,1137,2
2,2025-03-07,1137,2
3,2025-03-10,1137,2
4,2025-03-13,1137,2
5,2025-03-12,1137,2
6,2025-03-05,1137,2
7,2025-03-03,1137,2
8,2025-03-16,1137,2
9,2025-03-06,1137,2


In [12]:
con.sql("""
SELECT campaign_id, COUNT(*) as n
FROM campaigns
GROUP BY campaign_id
HAVING COUNT(*) > 1  
""")

┌─────────────┬───────┐
│ campaign_id │   n   │
│    int64    │ int64 │
└─────────────┴───────┘
        0 rows       

In [13]:
con.sql("""
SELECT week_start, COUNT(*) as n
FROM weekly_sales
GROUP BY week_start
HAVING COUNT(*) > 1  
""")

┌────────────┬───────┐
│ week_start │   n   │
│    date    │ int64 │
└────────────┴───────┘
        0 rows      

In [14]:
con.sql("""
SELECT week_start, COUNT(*) as n
FROM promotions
GROUP BY week_start
HAVING COUNT(*) > 1  
""")

┌────────────┬───────┐
│ week_start │   n   │
│    date    │ int64 │
└────────────┴───────┘
        0 rows      

In [15]:
con.sql("""
SELECT *
FROM daily_performance
WHERE campaign_id = 1137
  AND date BETWEEN '2025-03-03' AND '2025-03-16'
ORDER BY date
""").df()

,date,campaign_id,spend,impressions,clicks,conversions,revenue
0,2025-03-03,1137,606.44,10632,542,21,1655.04
1,2025-03-03,1137,606.44,10632,542,21,1655.04
2,2025-03-04,1137,650.22,12392,562,18,1369.80
3,2025-03-04,1137,650.22,12392,562,18,1369.80
4,2025-03-05,1137,508.97,8838,464,13,1078.01
5,2025-03-05,1137,508.97,8838,464,13,1078.01
6,2025-03-06,1137,579.30,9745,419,18,1553.05
7,2025-03-06,1137,579.30,9745,419,18,1553.05
8,2025-03-07,1137,593.00,10725,481,23,2010.06
9,2025-03-07,1137,593.00,10725,481,23,2010.06


- No duplicates in campaigns (by campaign_id), weekly_sales (by week_start), or promotions (by ).
- daily_performance: campaign 1137 has 14 days (2025-03-03 to 2025-03-16) duplicated as exact copies. Summing the raw table would double-count them. Fix: view `daily_clean` using SELECT DISTINCT. Raw table left untouched. Use daily_clean for all analysis.

In [16]:
con.sql("""
CREATE OR REPLACE VIEW daily_clean AS
SELECT DISTINCT * FROM daily_performance
""")

con.sql("SELECT COUNT(*) AS n_rows FROM daily_clean").df()

,n_rows
0,13048
